# NER & Relation Extraction — guided tour

This notebook walks through the `ner_re` package:

- **Part A**: cleaning, spaCy analysis, Word2Vec and BERT embeddings
- **Part B**: named entity recognition with spaCy and a CRF
- **Part C**: rule-based relations and knowledge graph, then supervised relation classifiers

Heavy training runs are better launched with the scripts in `scripts/`.

## Setup (Colab)
Skip this cell if you installed the package locally with `pip install -e .`.

In [ ]:
import os

if "COLAB_GPU" in os.environ or os.getcwd() == "/content":
    !git clone https://github.com/<your-username>/ner-relation-extraction.git
    %cd ner-relation-extraction
    !pip install -q -r requirements.txt
    !pip install -q -e .
    !python -m spacy download en_core_web_sm -q
    !python -m spacy download en_core_web_md -q

In [ ]:
%matplotlib inline

from ner_re.preprocessing import (build_text_sample, extract_entities, load_ner_dataset,
                                  load_spacy, spacy_preprocess, top_verbs)

NER_CSV = "data/ner_dataset.csv"
df = load_ner_dataset(NER_CSV)
text = build_text_sample(df, n_words=30_000)
nlp = load_spacy("en_core_web_sm")
df.head()

## Part A — Preprocessing and representations

In [ ]:
processed = spacy_preprocess(text, nlp)
print(len(processed["tokens"]), "tokens")
print(list(zip(processed["tokens"][:10], processed["pos"][:10])))
print(processed["lemmas"][:15])

In [ ]:
from ner_re.embeddings import bert_token_embeddings, train_word2vec

w2v = train_word2vec(text)
print(w2v.wv.most_similar("London", topn=5))

tokens, emb = bert_token_embeddings(text)
emb.shape

## Part B — Named entity recognition

In [ ]:
entities = extract_entities(text, nlp)
entities["label"].value_counts()

In [ ]:
from ner_re.crf import evaluate_ner, group_sentences, split_dataset, train_crf
from ner_re.visualization import plot_confusion_matrix

sentences, tags = group_sentences(df)
X_train, X_test, y_train, y_test = split_dataset(sentences, tags, test_size=0.2, seed=42)
crf = train_crf(X_train, y_train)
y_pred = crf.predict(X_test)

metrics = evaluate_ner(y_test, y_pred)
print(metrics["report"])

labels = sorted({t for s in y_test for t in s} - {"O"}) + ["O"]
plot_confusion_matrix([t for s in y_test for t in s], [t for s in y_pred for t in s], labels)

In [ ]:
i = 7
for feats, pred, gold in zip(X_test[i], y_pred[i], y_test[i]):
    print(f"{feats['word.lower()']:<15} pred={pred:<6} gold={gold}")

## Part C — Rule-based relations and knowledge graph

In [ ]:
from ner_re.coreference import resolve_document
from ner_re.rule_relations import build_knowledge_graph, extract_relations
from ner_re.visualization import plot_knowledge_graph

nlp_md = load_spacy("en_core_web_md")
print(top_verbs(text, nlp_md, top_n=15))

triples = extract_relations(text, nlp_md)
print(len(triples), "triples without coreference")

triples_coref = extract_relations(resolve_document(text, nlp_md), nlp_md)
print(len(triples_coref), "triples with coreference")
plot_knowledge_graph(build_knowledge_graph(triples_coref))

## Part C — Supervised relation classification

Training is long; run it with the script, e.g.

```bash
python scripts/train_relation_classifier.py --model bert     # NYT
python scripts/train_relation_classifier.py --model bilstm   # DocRED
```

Then load the trained models here.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from ner_re.bert_re import predict_relation as bert_predict, predict_text_relations

bert_dir = "outputs/bert_re/best_model"
tokenizer = AutoTokenizer.from_pretrained(bert_dir)
bert = AutoModelForSequenceClassification.from_pretrained(bert_dir)

print(bert_predict("Barack Obama was born in Honolulu .", "Barack Obama", "Honolulu", bert, tokenizer))
predict_text_relations(text[:3000], nlp, bert, tokenizer,
                       entity_types={"PERSON", "ORG", "GPE", "LOC"}, min_confidence=0.7)[:10]

In [ ]:
from ner_re.bilstm_re import load_bilstm, predict_relation as bilstm_predict
from ner_re.labels import property_name

model, vocab, id2label, cfg = load_bilstm("outputs/bilstm_re/best_model.pt")
label, conf = bilstm_predict("John is the father of Jack", "John", "Jack",
                             model, vocab, id2label, cfg["max_len"])
property_name(label), conf